In [1]:
import argparse
import torch
import cv2
import os
import numpy as np
from PIL import Image
from  matplotlib import pyplot as plt
from torchvision.transforms import Compose, Resize, ToTensor, Normalize
from torchvision.transforms import InterpolationMode
BICUBIC = InterpolationMode.BICUBIC

from segment_anything import sam_model_registry, SamPredictor
from numpy import savetxt
from numpy import genfromtxt
from numpy import linalg as la
from MaskGeneratorClass import MaskGenerator 
import tkinter as tk
from tkinter import PhotoImage
from tkinter import ttk, messagebox, filedialog
from PIL import Image, ImageTk
from pathlib import Path

# Open3D for point cloud processing and visualization
import open3d as o3d

from detection import RealsenseSubscriber
from process_point_cloud_baseline1 import pointCloud
import math

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
def run_image_clicker_app(image_path):
    coords = {"x": None, "y": None}  # Dictionary to store coordinates

    def get_click_coordinates(event):
        # Store coordinates and close the window
        coords["x"], coords["y"] = event.x, event.y
        root.destroy()  # This will close the Tkinter window

    root = tk.Tk()
    root.title("Click on Image")

    # Load the image
    image = Image.open(image_path)
    photo = ImageTk.PhotoImage(image)

    # Setup a canvas
    canvas = tk.Canvas(root, width=image.width, height=image.height)
    canvas.pack()

    # Display the image on the canvas
    canvas.create_image(0, 0, anchor=tk.NW, image=photo)

    # Bind the click event
    canvas.bind("<Button-1>", get_click_coordinates)

    root.mainloop()

    return coords["x"], coords["y"]  # Return the stored coordinates

def show_mask(mask, img, random_color=False, opacity=0.6):
    if random_color:
        color = np.concatenate([np.random.random(3)*255], axis=0)
    else:
        color = np.array([30, 144, 255])
    h, w = mask.shape[-2:]
    color_seg = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    fg_mask = mask != False
    
    img[fg_mask] = color_seg[fg_mask] * opacity
    
    return img

In [3]:
data_path="Spring_24_Data/"

extrinsics1 = data_path+"pose_1/"+"camera_pose.csv"
in_params = data_path+"pose_1/"+"intrinsic_params.csv"
in_model = data_path+"pose_1/"+"distortion_model.csv"
in_coeff = data_path+"pose_1/"+"intrinsic_coeffs.csv"
depImg = data_path+"pose_1/"+"depth_image_pixel_transform.png"
depArr = data_path+"pose_1/"+"depth_array.csv"
image_path = data_path+"pose_1/"+"cheezit_1.png"

rsObj = RealsenseSubscriber(in_params,in_model,in_coeff,depArr,depImg)

device = "cpu"

Params Loaded!


In [4]:
#1. Prepare Image For Inference 
print("Preparing Model + Images")
pil_img = Image.open(image_path) 

#2. Prepare Model For Inference
sam_checkpoint = "sam_vit_h_4b8939.pth"
model_type = "vit_h"
sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
sam.to(device=device)
predictor = SamPredictor(sam)
predictor.set_image(np.array(pil_img))
print("Model + Image Ready")


# Use the path to your image
x, y = run_image_clicker_app(image_path)
print(f"Clicked coordinates: ({x}, {y})")

temp =  MaskGenerator(predictor)
temp.setCoordinates(x,y)

masks = temp.mask

image = cv2.imread(data_path+"pose_"+str(1)+"/cheezit_"+str(1)+".png")

img_with_mask = show_mask(masks, image, False, 0.6)
cv2.imwrite(data_path+"1st clicked sam output "+str(1)+".png",img_with_mask)

Preparing Model + Images
Model + Image Ready
Clicked coordinates: (639, 311)


True

In [5]:
file_name = "cheeseit.csv"

for trans_cnt in range(2,9):

    extrinsics1 = data_path+"pose_"+str(trans_cnt-1)+"/"+"camera_pose.csv"
    in_params = data_path+"pose_"+str(trans_cnt-1)+"/"+"intrinsic_params.csv"
    in_model = data_path+"pose_"+str(trans_cnt-1)+"/"+"distortion_model.csv"
    in_coeff = data_path+"pose_"+str(trans_cnt-1)+"/"+"intrinsic_coeffs.csv"
    depImg = data_path+"pose_"+str(trans_cnt-1)+"/"+"depth_image_pixel_transform.png"
    depArr = data_path+"pose_"+str(trans_cnt-1)+"/"+"depth_array.csv"
    image_path = cv2.imread(data_path+"pose_"+str(trans_cnt-1)+"/cheezit_"+str(trans_cnt-1)+".png")


    print("save most recent mask to file")

    rsObj = RealsenseSubscriber(in_params,in_model,in_coeff,depArr,depImg)

    savetxt(file_name, masks, delimiter=',')

    # Call deproject project and transform code 
    extrinsinc_pos1 = genfromtxt(extrinsics1, delimiter=',')

    img_mask =  genfromtxt(file_name,delimiter=",")
    result_arr = rsObj.deproject_pixel_to_point(img_mask)








    ############### DEPROJECTED POINTS ########################
    # Initializing object to class pointCloud() for visualization purposes:
    cloud_object_deprojected_points = pointCloud()

    '''Rotation matrix and position vector for the robot base or world reference frame: '''
    cloud_object_deprojected_points.R_base = np.identity(3)
    cloud_object_deprojected_points.p_base = np.zeros([3,1])

    cloud_object_deprojected_points.g_base_cam = extrinsinc_pos1

    # Extracting the rotation matrix and position vector: 
    R_pose_1 = extrinsinc_pos1[0:3, 0:3]
    p_pose_1 = np.reshape(extrinsinc_pos1[0:3, 3], [3,1])

    cloud_object_deprojected_points.R_base_cam = R_pose_1
    cloud_object_deprojected_points.p_base_cam = p_pose_1

    num_points = len(result_arr)
    result_arr = np.reshape(np.asarray(result_arr), [num_points, 3])

    '''Creating a Open3d PointCloud Object for the cloud corresponding to just the bounding box'''
    objectCloud = o3d.geometry.PointCloud()
    objectCloud.points = o3d.utility.Vector3dVector(result_arr.astype(np.float64))
    objectCloud.paint_uniform_color([0, 0, 1])

    '''Visualizing just the CheezIt point cloud using open3D:'''
    #o3d.visualization.draw_geometries([objectCloud])

    cloud_object_deprojected_points.cloud = objectCloud

    '''Transforming the point cloud in the Panda base reference frame: '''
    cloud_object_deprojected_points.transformToBase()

    '''Visualizing the downsampled point cloud. '''
    print('Cloud transformed to base')
    #o3d.visualization.draw_geometries([cloud_object_deprojected_points.cloud])

    '''# Downsample it and inspect the normals'''
    cloud_object_deprojected_points.cloud = cloud_object_deprojected_points.cloud.voxel_down_sample(voxel_size=0.009)
    #cloud_object_deprojected_points.cloud = cloud_object_deprojected_points.cloud.uniform_down_sample(every_k_points=100)


    '''This needs to commented out when dealing with objects like the spatula and screw driver'''
    cloud_object_deprojected_points.removePlaneSurface()

    '''# Visualizing the downsampled point cloud. '''
    print('Plane surface removed!')
    #o3d.visualization.draw_geometries([cloud_object_deprojected_points.cloud])

    '''Specifying parameters for DBSCAN Clustering:
    Just like the parameters for downsampling even the parameters for DBSCAN Clustering are dependent on the 
    units used computing and extracting the point cloud data.'''
    cloud_object_deprojected_points.eps = 0.02
    cloud_object_deprojected_points.min_points = 10
    cloud_object_deprojected_points.getObjectPointCloud()








    print("New transofrmed Image "+str(trans_cnt))
    new_cloud_object_deprojected_points = cloud_object_deprojected_points
    extrinsics2 = data_path+"pose_"+str(trans_cnt)+"/camera_pose.csv"
    in_params2 = data_path+"pose_"+str(trans_cnt)+"/intrinsic_params.csv"
    in_model2 = data_path+"pose_"+str(trans_cnt)+"/distortion_model.csv"
    in_coeff2 = data_path+"pose_"+str(trans_cnt)+"/intrinsic_coeffs.csv"
    image2 = cv2.imread(data_path+"pose_"+str(trans_cnt)+"/cheezit_"+str(trans_cnt)+".png")

    transformed_coords = ""
    cnt = 0


    # Call deproject project and transform code 
    extrinsinc_pos2 = genfromtxt(extrinsics2, delimiter=',')

    ############### TRANSFORMING THE POINTS (UPDATED) ###############

    # Extracting the deprojected points which have been transformed in the base reference frame: 
    new_cloud_object_deprojected_points.points = np.asarray(new_cloud_object_deprojected_points.processed_cloud.points)


    print('Shape of the deprojected points: ', new_cloud_object_deprojected_points.points.shape)

    # Transforming these deprojected points from the base reference frame to the second camera pose:
    # Initializing object to class pointCloud() for visualization purposes:
    cloud_object_transformed_points = pointCloud()

    '''Rotation matrix and position vector for the robot base or world reference frame: '''
    cloud_object_transformed_points.R_base = np.identity(3)
    cloud_object_transformed_points.p_base = np.zeros([3,1])

    cloud_object_transformed_points.g_base_cam = extrinsinc_pos2

    # Extracting the rotation matrix and position vector: 
    R_pose_2 = extrinsinc_pos2[0:3, 0:3]
    R_pose_2_inv = la.inv(R_pose_2)
    p_pose_2 = np.reshape(extrinsinc_pos2[0:3, 3], [3,1])

    cloud_object_transformed_points.R_base_cam = R_pose_2
    cloud_object_transformed_points.p_base_cam = p_pose_2

    # transformed_points_updated = []
    transformed_points_updated = np.zeros([new_cloud_object_deprojected_points.points.shape[0], new_cloud_object_deprojected_points.points.shape[1]])









    # Implementation with homogeneous coordinates: 
    point_h = np.ones([4,1])
    for i in range(new_cloud_object_deprojected_points.points.shape[0]):
        point_h[0,:] = new_cloud_object_deprojected_points.points[i, 0]
        point_h[1,:] = new_cloud_object_deprojected_points.points[i, 1]
        point_h[2,:] = new_cloud_object_deprojected_points.points[i, 2]
        extrinsinc_pos2_inv = la.inv(extrinsinc_pos2)
        result = np.matmul(extrinsinc_pos2_inv, point_h)
        transformed_points_updated[i,:] = np.reshape(result[0:3, :], [1,3])
        transformed_pixel = rsObj.project_point_to_pixel(result[0:3,:],in_params2,in_model2,in_coeff2)

        if not math.isnan(transformed_pixel[0])  and not math.isnan(transformed_pixel[1]):
            transformed_coords += str(round(transformed_pixel[0])) + " " + str(round(transformed_pixel[1]))+ "\n"
        cnt+=1

    f2 = open("transformed_points.txt","w")
    f2.write(transformed_coords)
    f2.close()

    image = image2.copy()
    f = open("transformed_points.txt","r")

    minx = 2000
    miny= 2000
    maxx = -1
    maxy = -1
    for line in f:
        line = line.strip("\n")
        x, y = line.split(" ")
        cv2.circle(image, (int(x), int(y)), 3, (255, 0, 0), 3) 

        minx = min(int(x),minx)
        miny= min(int(y),miny)
        maxx = max(int(x),maxx)
        maxy = max(int(y),maxy)

    cv2.imwrite(data_path+"Sampled from "+str(trans_cnt-1)+" img and Transformed to "+str(trans_cnt)+" img"+".png",image)



    image = image2.copy()
    image_path = data_path+"pose_"+str(trans_cnt)+"/cheezit_"+str(trans_cnt)+".png"
    pil_img = Image.open(image_path)

    predictor.set_image(np.array(pil_img))

    box = np.array([minx,miny,maxx,maxy])


    masks, scores, _ = predictor.predict(   
            box = box[None, :],
            multimask_output=True,
            )   
    img_with_mask = show_mask(masks[np.argmax(scores)], image, False, 0.6)
    
    masks=masks[np.argmax(scores)]


    cv2.imwrite(data_path+" SAM output img "+str(trans_cnt)+".png",img_with_mask)

save most recent mask to file
Params Loaded!
720 1280
Cloud transformed to base
Plane surface removed!
[Open3D DEBUG] Precompute neighbors.
[Open3D DEBUG] Done Precompute neighbors.
[Open3D DEBUG] Compute Clusters
[Open3D DEBUG] Done Compute Clusters: 1
point cloud has 1 clusters
New transofrmed Image 2
Shape of the deprojected points:  (974, 3)
save most recent mask to file
Params Loaded!
720 1280
Cloud transformed to base
Plane surface removed!
[Open3D DEBUG] Precompute neighbors.
[Open3D DEBUG] Done Precompute neighbors.
[Open3D DEBUG] Compute Clusters
[Open3D DEBUG] Done Compute Clusters: 3
point cloud has 3 clusters
New transofrmed Image 3
Shape of the deprojected points:  (998, 3)
save most recent mask to file
Params Loaded!
720 1280
Cloud transformed to base
Plane surface removed!
[Open3D DEBUG] Precompute neighbors.
[Open3D DEBUG] Done Precompute neighbors.
[Open3D DEBUG] Compute Clusters
[Open3D DEBUG] Done Compute Clusters: 1
point cloud has 1 clusters
New transofrmed Image 4